# పాఠం 01 - AI ఏజెంట్స్ పరిచయం

**AI ఏజెంట్స్ ప్రారంభికులకు** కోర్సులో తొలి పాఠానికి స్వాగతం!

ఒక **AI ఏజెంట్** అనేది పెద్ద భాషా మోడల్ (LLM) ని దాని తర్కాన్ని ఇంజిన్ గా ఉపయోగించే ప్రోగ్రాం మరియు నిజ జీవితం లో *చర్యలు* తీసుకోవచ్చు — APIలను పిలవడం, డేటాబేస్‌లను విచారణ చేయడం, లేదా కోడ్ నడిపించడం — వాడుకరికి పక్షపాతంగా ఒక లక్ష్యాన్ని సాధించడానికి.

ఈ నోటుబుక్‌లో మీరు మీ మొదటి ఏజెంట్‌ను నిర్మిస్తారు: ఒక **ప్రయాణ ఏజెంట్** ఇది సెలవు గమ్యస్థలాలను సూచిస్తుంది. దారిలో మీరు నేర్చుకుంటారు:

1. **Microsoft Agent Framework** ఉపయోగించి Azure AI Foundry Agent సేవకు కనెక్ట్ చేయడం.
2. ఏజెంట్‌కు ఒక **టూల్** ఇవ్వడం — ఆహ్లాదకరమైన పాథాన్ ఫంక్షన్‌ను ఇది పిలవగలదు.
3. ఏజెంట్‌ను నడపడం మరియు దాని ప్రతిస్పందనను పరిశీలించడం.
4. ఏజెంట్ యొక్క ప్రతిస్పందన టోకెన్ వారీగా స్ట్రీమ్ చేయడం.


## అమరిక

ఈ నోట్‌బుక్‌ను నడపడానికి ముందు, మీరు కలిగి ఉన్నదీ నిర్ధారించుకోండి:

1. **డిప్లాయ్ చేయబడిన చాట్ మోడల్‌తో కూడిన Azure AI Foundry ప్రాజెక్టు** (ఉదాహరణకు `gpt-4o-mini`).
2. **Azure CLI తో లాగిన్ అయ్యి ఉన్నారు** — మీ టెర్మినల్‌లో `az login` నడపండి.
3. **అవసరమైన వాతావరణ చరాల‌ను సెట్ చేసుకున్నాయి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ Azure AI Foundry ప్రాజెక్టు ఎండ్‌పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ డిప్లాయ్ చేసిన మోడల్ పేరు.

కింద ఉన్న సెల్ మీరు అవసరమైన Python ప్యాకేజీలను ఇన్‌స్టాల్ చేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మీ మొదటి ఏజెంట్‌ను సృష్టించడం

ఏజెంట్‌కు రెండు విషయాలు అవసరం:

- **సూచనలు** అది *ఎవరంటూ* మరియు *ఎలా ప్రవర్తించాలో* చెప్తాయి (సిస్టమ్ ప్రాంప్ట్).
- **టూల్స్** — ఏజెంట్ సమాచారం సేకరించడానికి లేదా చర్యలు తీసుకోవడానికి పిలవగల Python ఫంక్షన్లు, ఇవి `@tool` తో డికోరేట్ చేయబడతాయి.

కింద, ప్రజాదరణ ఉన్న సెలవుల గమ్యస్థలాల జాబితాను తిరిగి ఇచ్చే ఒక సాదారణ టూల్‌ను నిర్వచించాము. యూజర్ ప్రయాణ సిఫారసులు అడిగితే ఏజెంట్ ఈ టూల్‌ను ఉపయోగిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## స్ట్రీమింగ్ ప్రతిస్పందనలు

ఇంకాస్త పరస్పర చర్య అనుభవానికి మీరు ఏజెంట్ ప్రతిస్పందనను **స్ట్రీమ్** చేయవచ్చు. పూర్తి సమాధానానికి ఎదురుచూడకుండా, ఏజెంట్ రూపొందిస్తున్నట్లుగా టెక్స్ట్ భాగాలను అందిస్తుంది. ఇది ప్రత్యేకంగా చాట్ ఇంటర్‌ఫేస్‌లలో ఉపయోగకరం, అక్కడ మీరు అవుట్పుట్‌ను రియల్ టైంలో ప్రదర్శించాలనుకుంటారు.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్న విషయాలు:

- **ప్రొవైడర్‌ని సృష్టించడం** `AzureAIProjectAgentProvider` ద్వారా Azure AI Foundry Agent Serviceకి కనెక్ట్ అవుతుంది.
- **టూల్‌ని నిర్వచించడం** `@tool` డెకొరేటర్ ఉపయోగించి, తద్వారా ఏజెంట్ మీ Python ఫంక్షన్లు పిలవవచ్చు.
- **ఏజెంట్‌ని రన్ చేయడం** యూజర్ సందేశంతో మరియు దాని స్పందనను ప్రింట్ చేయడం.
- **ప్రతిస్పందనలను స్ట్రీమ్ చేయడం** రియల్-టైమ్ అవుట్‌పుట్ కోసం.

తరువాతి పాఠంలో మనం ఏజెంటిక్ ఫ్రేమ్‌వర్క్‌లను మరింత లోతుగా పరిశీలించి, ఏజెంట్లకు మరింత శక్తివంతమైన టూల్స్ మరియు బహుళ-దశ తర్క శక్తులను ఎలా ఇవ్వాలో నేర్చుకుంటాం.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**నిరాకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వాన్ని సాధించేందుకు శ్రద్ధ వహిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో తప్పులుంటే లేదా తప్పులుంటే ఉండవచ్చు. స్థానిక భాషలోని మూల పత్రం అధికారిక మూలంగా పరిగణించాలి. సమగ్ర సమాచారం కోసం, నిపుణుల మానవ అనువాదం సూచించబడుతుంది. ఈ అనువాదం వలన కలిగే ఏవైనా అపార్థాలు లేదా తప్పివిప్రతిపాదనలకి మేము బాధ్యత తీసుకోము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
